In [4]:
import time
import re
import csv
import requests
import pandas as pd
from dateutil import parser
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

In [5]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'
}

base_url = 'https://www.eventbrite.ca/d/canada/events--this-month/?page={}'
max_workers = 10
max_empty_pages = 3  # Stop scraping after this many empty pages in a row
max_retries = 3

def scrape_page(page):
    for attempt in range(max_retries):
        try:
            print(f"Scraping page {page} (Attempt {attempt + 1})...")
            url = base_url.format(page)
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            ul = soup.find('ul', class_='SearchResultPanelContentEventCardList-module__eventList___2wk-D')
            if not ul:
                return [], page

            events = []
            for li in ul.find_all('li'):
                h3 = li.find('h3')
                a = li.find('a', href=True)
                title = h3.get_text(strip=True) if h3 else 'No title'
                link = a['href'] if a else 'No link'
                events.append({'title': title, 'link': link})

            return events, page
        except Exception as e:
            print(f"Error on page {page}, attempt {attempt + 1}: {e}")
            time.sleep(1)
    return [], page

# Dynamic page crawl with concurrency
all_events = []
page = 1
empty_count = 0

while empty_count < max_empty_pages:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        pages_to_scrape = list(range(page, page + max_workers))
        futures = {executor.submit(scrape_page, p): p for p in pages_to_scrape}

        for future in as_completed(futures):
            events, p = future.result()
            if events:
                all_events.extend(events)
                print(f"✅ Page {p} returned {len(events)} events.")
                empty_count = 0  # Reset empty counter on success
            else:
                print(f"⚠️ Page {p} returned no events.")
                empty_count += 1

    page += max_workers

# Save results to CSV
with open('events_data_links.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['title', 'link'])
    writer.writeheader()
    writer.writerows(all_events)

print(f"✅ Scraping completed. {len(all_events)} events saved to 'events_data_links.csv'.")


Scraping page 1 (Attempt 1)...Scraping page 2 (Attempt 1)...

Scraping page 3 (Attempt 1)...
Scraping page 4 (Attempt 1)...
Scraping page 5 (Attempt 1)...
Scraping page 6 (Attempt 1)...
Scraping page 7 (Attempt 1)...
Scraping page 8 (Attempt 1)...
Scraping page 9 (Attempt 1)...
Scraping page 10 (Attempt 1)...
✅ Page 1 returned 20 events.
✅ Page 2 returned 20 events.
✅ Page 10 returned 20 events.
✅ Page 8 returned 20 events.
✅ Page 3 returned 20 events.
✅ Page 4 returned 20 events.
✅ Page 7 returned 20 events.
✅ Page 6 returned 20 events.
✅ Page 9 returned 20 events.
✅ Page 5 returned 20 events.
Scraping page 11 (Attempt 1)...
Scraping page 12 (Attempt 1)...
Scraping page 13 (Attempt 1)...
Scraping page 14 (Attempt 1)...
Scraping page 15 (Attempt 1)...
Scraping page 16 (Attempt 1)...
Scraping page 17 (Attempt 1)...
Scraping page 18 (Attempt 1)...
Scraping page 19 (Attempt 1)...
Scraping page 20 (Attempt 1)...
✅ Page 11 returned 20 events.
✅ Page 15 returned 20 events.
✅ Page 16 returned

In [6]:
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip() if text else ''

def get_text_safe(soup, selector, attr='text', default=''):
    el = soup.select_one(selector)
    if not el:
        return default
    return clean_text(el.get_text()) if attr == 'text' else el.get(attr, default)

def extract_ticket_price(soup):
    # Try primary price method
    ticket_price = get_text_safe(soup, '[data-testid="panel-info"]').replace('From', '').strip()
    
    # Fallback if primary fails
    if not ticket_price:
        price_tag = soup.select_one('.CondensedConversionBar-module__priceTag___3AnIu')
        if price_tag:
            ticket_price = clean_text(price_tag.text.replace("CA$", "$"))
    return ticket_price

def extract_event_data(row):
    url = row['link']
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"⚠️ Failed to fetch {url} - Status: {response.status_code}")
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract fields
        date = get_text_safe(soup, 'time.start-date')
        title = get_text_safe(soup, 'h1.event-title')
        host = get_text_safe(soup, '[data-testid="organizer-info-details"] strong')
        full_date_time = get_text_safe(soup, '.date-info__full-datetime')
        event_place = get_text_safe(soup, '.location-info__address-text')

        address_div = soup.select_one('.location-info__address')
        if address_div:
            full_address_text = address_div.get_text(separator=' ')
            cleaned_address = clean_text(full_address_text.replace(event_place, ''))
            full_address = cleaned_address.split('M5S')[0].strip() if 'M5S' in cleaned_address else cleaned_address
        else:
            full_address = ''

        postal_code_match = re.search(r'[A-Z]\d[A-Z] ?\d[A-Z]\d', address_div.text if address_div else '')
        postal_code = postal_code_match.group(0) if postal_code_match else ''

        # Use improved ticket price extraction
        ticket_price = extract_ticket_price(soup)

        # Extract image URL
        image_tag = soup.select_one('picture[data-testid="hero-image"] img')
        image_url = image_tag['src'] if image_tag and 'src' in image_tag.attrs else ''

        return {
            "Date": date,
            "Title": title,
            "Host": host,
            "Full_date_time": full_date_time,
            "Event_place": event_place,
            "Full_address": full_address,
            "Postal_code": postal_code,
            "Ticket_price": ticket_price,
            "Image_url": image_url,
            "Event_link": url
        }

    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")
        return None

# Load input CSV
df = pd.read_csv('events_data_links.csv')

# Use ThreadPoolExecutor for concurrent requests
results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(extract_event_data, row) for _, row in df.iterrows()]
    for future in as_completed(futures):
        result = future.result()
        if result:
            results.append(result)

# Save results
output_df = pd.DataFrame(results)
output_df.to_csv("events.csv", index=False)

print("✅ Parallel scraping complete. Data saved to 'events.csv'.")


✅ Parallel scraping complete. Data saved to 'events.csv'.


In [7]:
# Load the CSV
df = pd.read_csv("events.csv")

# --- 1. Extract start_date, end_date, start_time, end_time from 'Full_date_time' ---

def extract_datetime_parts(text):
    text = str(text)
    try:
        # Normalize dashes and spacing
        text = text.replace('–', '-').replace('â€“', '-')
        text = re.sub(r'\s*-\s*', ' - ', text)

        # Split on dash to separate ranges
        parts = text.split(' - ')
        start_part = parts[0].strip()
        end_part = parts[1].strip() if len(parts) > 1 else ''

        # Try parsing start datetime
        start_dt = parser.parse(start_part, fuzzy=True)
        start_date = start_dt.strftime('%Y-%m-%d')
        start_time = start_dt.strftime('%H:%M') if start_dt.time() != parser.parse("00:00").time() else ''

        # Try parsing end datetime
        try:
            end_dt = parser.parse(end_part, fuzzy=True)
            end_date = end_dt.strftime('%Y-%m-%d')
            end_time = end_dt.strftime('%H:%M') if end_dt.time() != parser.parse("00:00").time() else ''
        except:
            end_date = start_date
            end_time = ''
    except:
        start_date = end_date = start_time = end_time = ''

    return pd.Series([start_date, end_date, start_time, end_time])

df[['start_date', 'end_date', 'start_time', 'end_time']] = df['Full_date_time'].apply(extract_datetime_parts)

# --- 2. Extract 'state_name' from 'Full_address' ---

def extract_state(address):
    try:
        parts = address.split(',')
        if len(parts) >= 2:
            state_part = parts[1].strip()
            return state_part[:2].upper()
    except:
        return ''
    return ''

df['state_name'] = df['Full_address'].apply(extract_state)

# --- 3. Clean 'Ticket_price' column ---

def clean_ticket_price(price):
    if pd.isna(price):
        return ''
    price = str(price)

    # Normalize and remove encoding artifacts
    price = price.replace('â€“', '-').replace('–', '-').replace('Â', '')
    price = price.replace('CA$', '').replace('CA', '').replace('$', '')
    price = price.strip()

    # Handle known keywords
    keywords = ['sold out', 'free', 'donation']
    if any(kw in price.lower() for kw in keywords):
        return price.title()

    # Remove unwanted characters
    price = re.sub(r'[^0-9.\- ]', '', price)

    return price.strip()

df['ticket_price'] = df['Ticket_price'].apply(clean_ticket_price)

# --- 4. Rename/Map required columns to match final schema ---

df.rename(columns={
    'Title': 'event_title',
    'Host': 'event_host',
    'Image_url': 'image_url',
    'Event_link': 'booking_url',
    'Event_place': 'event_place',
    'Full_address': 'full_address',
    'Postal_code': 'postal_code'
}, inplace=True)

# Fill other required columns with default or blank values
df['summary'] = ''
df['language'] = ''
df['event_type'] = ''
df['country_name'] = 'Canada'

# Ensure final column order
final_columns = [
    'event_title', 'summary', 'image_url', 'language', 'event_type', 'event_host',
    'ticket_price', 'booking_url', 'start_date', 'end_date', 'start_time', 'end_time',
    'event_place', 'full_address', 'country_name', 'state_name', 'city_name', 'postal_code'
]

# Add missing columns with default value
for col in final_columns:
    if col not in df.columns:
        df[col] = ''

# Reorder and save to new CSV
df[final_columns].to_csv("events_cleaned.csv", index=False)


C:\Users\ridha\AppData\Roaming\Python\Python312\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname EDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
C:\Users\ridha\AppData\Roaming\Python\Python312\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname MDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "
C:\Users\ridha\AppData\Roaming\Python\Python312\site-packages\dateutil\parser\_parser.py:1207: UnknownTimezoneWarning: tzname PDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will rais